In [1]:
# Add project root (the folder that contains "src/") to sys.path
import sys
import time
from pathlib import Path

def add_project_root(marker_dir="src", max_hops=5):
    p = Path.cwd().resolve()
    for _ in range(max_hops):
        if (p / marker_dir).exists():
            sys.path.insert(0, str(p))
            print(f"[OK] Added to sys.path: {p}")
            return
        p = p.parent
    raise RuntimeError(f"Could not find '{marker_dir}' within {max_hops} parents from {Path.cwd()}")

add_project_root()  

[OK] Added to sys.path: D:\Data\Projects\Thesis\Adaptive-Hierarchical-Feature-Modulation-U-Net-Model-for-Retinal-Vessel-Segmentation


In [2]:
# --- 0) Setup & imports ---
import os, sys, json
import numpy as np
from PIL import Image  # or imageio.v2
# If the package folder is next to the notebook; adjust if needed:
sys.path.append(".")

from src.retina_biomarkers import (
    to_bool_mask, skeletonize_mask, distance_transform, build_skeleton_graph,
    sample_width_along_skeleton, sample_widths_orthogonal, 
    area_density, length_density, caliber_stats,
    tortuosity_stats, fractal_dimension_boxcount,
    junction_metrics, branching_and_bifurcation_angles,
    branching_angles_roi,                         
    gap_metrics, metrics_by_rings
)


In [3]:

# --- 1) I/O helper: load a binary mask from a path ---
def load_binary_mask(mask_path: str) -> np.ndarray:
    """
    Loads a 2D binary vessel mask (0/1 or 0/255).
    If the image has multiple channels, it converts to L (grayscale).
    Any positive value becomes vessel.
    """
    if not os.path.exists(mask_path):
        raise FileNotFoundError(mask_path)
    img = Image.open(mask_path)
    if img.mode != "L":
        img = img.convert("L")
    arr = np.array(img)          # uint8
    mask = (arr > 0).astype(np.uint8)
    return mask

# --- 2) Core: compute all metrics from a binary mask ---
def compute_biomarkers_from_mask_path(
    mask_path: str,
    *,
    disc_center=None,      # (y, x) in px
    PD_px=None,            # optic disc diameter in px
    max_gap_px: int = 10,
    angle_k_ahead: int = 3,
    ortho_step: float = 0.5,     # sampling step along normal (px)
    ortho_max_radius: float = 20 # max half-width to search (px)
) -> dict:
    """Return a nested dict with global, topology, and (optional) ring metrics."""
    mask = load_binary_mask(mask_path)
    H, W = mask.shape

    # --- Geometry ---
    skel  = skeletonize_mask(mask)
    dist  = distance_transform(mask)
    graph = build_skeleton_graph(skel)

    # --- Calibre sampling (two flavors) ---
    widths_edt  = sample_width_along_skeleton(dist, graph)                 # VC from 2*EDT
    widths_orth = sample_widths_orthogonal(                                # VC from orthogonal chords (paper-consistent)
        mask, graph, k_tangent=3, step=ortho_step, max_radius=ortho_max_radius
    )

    # --- Global metrics ---
    global_metrics = {
        "area_density":          float(area_density(mask)),
        "length_density_px_inv": float(length_density(graph, mask.shape)),
        "fractal_dimension":     float(fractal_dimension_boxcount(skel)),
        # Tortuosity (curvature-based, length-weighted)
        **tortuosity_stats(graph),
        # Calibre (report BOTH; keep EDT for continuity, but orthogonal is what papers prefer)
        "vc_edt":  caliber_stats(widths_edt),     # -> median_width, iqr_width, frac_thin_len/med/thick
        "vc_orth": caliber_stats(widths_orth),
    }

    # (optional) convenience: mirror old flat keys to the orthogonal values so legacy code doesn’t break
    global_metrics["median_width"] = global_metrics["vc_orth"]["median_width"]
    global_metrics["iqr_width"]    = global_metrics["vc_orth"]["iqr_width"]

    # --- Topology & continuity ---
    topo = {}
    topo.update(junction_metrics(graph, mask.shape))
    topo.update(branching_and_bifurcation_angles(graph, k_ahead=angle_k_ahead))
    topo.update(gap_metrics(mask, graph, max_gap_px=max_gap_px))

    # Angles within 2 PD (paper-aligned), only if disc info is provided
    angles_2PD = None
    if disc_center is not None and PD_px is not None:
        angles_2PD = branching_angles_roi(
            graph, disc_center=disc_center, PD_px=PD_px, max_PD=2.0, k_ahead=angle_k_ahead
        )
    topo["angles_2PD"] = angles_2PD

    # --- Ring-wise metrics (use orth widths to match paper’s VC measurement) ---
    rings = None
    if disc_center is not None and PD_px is not None:
        rings = metrics_by_rings(
            mask, graph, widths_orth, disc_center=disc_center, PD_px=PD_px
        )

    return {
        "image_path": mask_path,
        "image_shape": (H, W),
        "global": global_metrics,
        "topology": topo,
        "rings": rings,  # None if not computed
    }

In [4]:
# --- 3) Example usage ---
# Replace with your actual mask path (PNG/TIF/JPG containing a binary vessel mask)
mask_path = "../../data/raw/DRIVE/test/1st_manual/01_manual1.png"

# If you know disc center and disc diameter in pixels, pass them; otherwise omit
# Example placeholders:
disc_center = (256, 256)  # (y, x)
PD_px = 100.0             # optic disc diameter in pixels

results = compute_biomarkers_from_mask_path(
    mask_path,
    disc_center=disc_center,   # or None
    PD_px=PD_px,               # or None
    max_gap_px=12,
    angle_k_ahead=3
)

# Pretty-print / save
print(json.dumps(results, indent=2))
# with open("biomarkers.json", "w") as f: json.dump(results, f, indent=2)

{
  "image_path": "../../data/raw/DRIVE/test/1st_manual/01_manual1.png",
  "image_shape": [
    584,
    565
  ],
  "global": {
    "area_density": 0.08922293611346829,
    "length_density_px_inv": 0.03426140733438053,
    "fractal_dimension": 1.406667044447461,
    "tortuosity_mean": 0.16643938422203064,
    "vc_edt": {
      "median_width": 2.8284270763397217,
      "iqr_width": 2.0,
      "frac_thin_len": 0.7728292971057295,
      "frac_med_len": 0.21760189013585352,
      "frac_thick_len": 0.009568812758417011
    },
    "vc_orth": {
      "median_width": 4.0,
      "iqr_width": 4.0,
      "frac_thin_len": 0.5781692068996989,
      "frac_med_len": 0.29049922424021174,
      "frac_thick_len": 0.13133156886008945
    },
    "median_width": 4.0,
    "iqr_width": 4.0
  },
  "topology": {
    "junction_count": 714.0,
    "endpoint_count": 133.0,
    "junction_density": 0.0021638986543823496,
    "endpoint_density": 0.00040307916111043763,
    "angle_mean": 110.42301940917969,
    "angle

In [6]:
# --- 3) Example usage ---
# Replace with your actual mask path (PNG/TIF/JPG containing a binary vessel mask)
mask_path = "../../data/raw/DRIVE/test/1st_manual/03_manual1.png"

# If you know disc center and disc diameter in pixels, pass them; otherwise omit
# Example placeholders:
disc_center = (256, 256)  # (y, x)
PD_px = 100.0             # optic disc diameter in pixels

results = compute_biomarkers_from_mask_path(
    mask_path,
    disc_center=disc_center,   # or None
    PD_px=PD_px,               # or None
    max_gap_px=12,
    angle_k_ahead=3
)

# Pretty-print / save
print(json.dumps(results, indent=2))
# with open("biomarkers.json", "w") as f: json.dump(results, f, indent=2)

{
  "image_path": "../../data/raw/DRIVE/test/1st_manual/03_manual1.png",
  "image_shape": [
    584,
    565
  ],
  "global": {
    "area_density": 0.09968784095041823,
    "length_density_px_inv": 0.03404832492598419,
    "fractal_dimension": 1.4000056827047025,
    "tortuosity_mean": 0.16090147197246552,
    "vc_edt": {
      "median_width": 2.8284270763397217,
      "iqr_width": 3.6568541526794434,
      "frac_thin_len": 0.6530219118709367,
      "frac_med_len": 0.3068865880086684,
      "frac_thick_len": 0.0400915001203949
    },
    "vc_orth": {
      "median_width": 4.0,
      "iqr_width": 5.0,
      "frac_thin_len": 0.5063405797101449,
      "frac_med_len": 0.30144927536231886,
      "frac_thick_len": 0.19221014492753624
    },
    "median_width": 4.0,
    "iqr_width": 5.0
  },
  "topology": {
    "junction_count": 799.0,
    "endpoint_count": 116.0,
    "junction_density": 0.002421505637046915,
    "endpoint_density": 0.00035155776457752457,
    "angle_mean": 110.93865966796875